# Alpamayo 1.5: Front Camera Projection + BEV Cross Check

This notebook:
1. Automatically reads front-camera calibration txt files from `repo_root/calibration/`
2. Runs offline inference on a local clip
3. Projects **History / GT / Pred** trajectories onto the **original front camera image**
4. Draws a **BEV trajectory plot** side-by-side with the image projection for easy correctness checking

Notes:
- The current validated pipeline uses notebook-side eval rotation `EVAL_XY_ROTATION_DEG = -90.0`
- Before projection, plotting-frame trajectories are converted back into raw t0-local xyz
- Projection is performed on the original decoded front-camera frame, not the resized model input image


In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('cwd =', Path.cwd().resolve())
print('repo_root =', repo_root)
print('src_path =', src_path)
print('src exists =', src_path.exists())
print('PYTHONPATH =', os.environ.get('PYTHONPATH'))

import av
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import scipy.spatial.transform as spt
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    load_offline_dataset,
    get_or_build_offline_clip_cache,
    _load_ego_pose_log,
    _lookup_pose_samples,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    rotate_xyz_xy_plane,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)


In [ ]:
SEED = 42
set_reproducible_seeds(SEED)
print('SEED =', SEED)


In [ ]:
# ===== Local paths =====
CLIP_DIR = Path('/workspace/dataset/')
CALIB_DIR = repo_root / 'calibration'
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Calibration txt files under repo_root/calibration =====
FRONT_K_PATH = CALIB_DIR / 'Front_camera_intrinsic.txt'
FRONT_K_ORIG_PATH = CALIB_DIR / 'Front_camera_original_intrinsic.txt'
FRONT_EXT_PATH = CALIB_DIR / 'Front_camera_extrinsics.txt'
FRONT_DIST_PATH = CALIB_DIR / 'Front_camera_distortion.txt'

# ===== Inference config =====
DEVICE = 'cuda'
T0_SOD = 175500.23
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)

# Keep the current validated eval-side plotting convention
EVAL_XY_ROTATION_DEG = -90.0

# If front camera row in sorted camera_indices is known, set it manually.
# Otherwise a heuristic finder below will be used.
FRONT_CAMERA_ROW_OVERRIDE = None

# Which sampled image frame to visualize on the front camera timeline.
# -1 means the latest sampled image frame aligned to t0.
FRONT_FRAME_T = -1

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('CLIP_DIR =', CLIP_DIR)
print('CALIB_DIR =', CALIB_DIR)
print('FRONT_K_PATH =', FRONT_K_PATH)
print('FRONT_K_ORIG_PATH =', FRONT_K_ORIG_PATH)
print('FRONT_EXT_PATH =', FRONT_EXT_PATH)
print('FRONT_DIST_PATH =', FRONT_DIST_PATH)
print('T0_SOD =', T0_SOD)
print('EVAL_XY_ROTATION_DEG =', EVAL_XY_ROTATION_DEG)


### Helper functions


In [ ]:
def load_matrix_txt(path: Path):
    arr = np.loadtxt(path, dtype=np.float64)
    print(f'Loaded {path.name}: shape={arr.shape}')
    return arr


def transform_points_homogeneous(points_xyz: np.ndarray, T: np.ndarray) -> np.ndarray:
    pts_h = np.concatenate(
        [points_xyz.astype(np.float64), np.ones((len(points_xyz), 1), dtype=np.float64)],
        axis=1,
    )
    out_h = (T @ pts_h.T).T
    return out_h[:, :3]


def project_points_pinhole(points_cam: np.ndarray, K: np.ndarray):
    X = points_cam[:, 0]
    Y = points_cam[:, 1]
    Z = points_cam[:, 2]

    valid = Z > 1e-6
    u = np.full_like(X, np.nan, dtype=np.float64)
    v = np.full_like(Y, np.nan, dtype=np.float64)

    u[valid] = K[0, 0] * X[valid] / Z[valid] + K[0, 2]
    v[valid] = K[1, 1] * Y[valid] / Z[valid] + K[1, 2]
    return np.stack([u, v], axis=-1), valid


def decode_single_frame(video_path: Path, frame_index: int) -> np.ndarray:
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for idx, frame in enumerate(container.decode(stream)):
            if idx == frame_index:
                return frame.to_ndarray(format='rgb24')
    raise IndexError(f'Frame index {frame_index} out of range for {video_path}')


def pick_front_camera_row(data, clip_dir: Path, row_override=None):
    if row_override is not None:
        return int(row_override)

    camera_indices = data['camera_indices'].cpu().tolist()
    camera_names = [helper.CAMERA_DISPLAY_NAMES.get(i, f'Camera {i}') for i in camera_indices]

    for row, name in enumerate(camera_names):
        lower = name.lower()
        if 'front' in lower:
            print(f'Heuristic front camera row matched by display name: row={row}, name={name}')
            return row

    camera_paths = helper.discover_offline_camera_files(clip_dir)
    camera_paths_sorted = sorted(camera_paths, key=lambda p: helper.infer_camera_index(p.name))
    for row, p in enumerate(camera_paths_sorted):
        lower = p.name.lower()
        if 'front' in lower:
            print(f'Heuristic front camera row matched by filename: row={row}, file={p.name}')
            return row

    print('WARNING: Could not confidently identify front camera row; defaulting to row 0')
    return 0


def choose_reasonable_extrinsic_direction(points_ego_global: np.ndarray, T_candidate: np.ndarray, K: np.ndarray, image_shape_hw):
    h, w = image_shape_hw

    def score(T):
        pts_cam = transform_points_homogeneous(points_ego_global, T)
        uv, valid_z = project_points_pinhole(pts_cam, K)
        valid_uv = (
            valid_z &
            np.isfinite(uv[:, 0]) & np.isfinite(uv[:, 1]) &
            (uv[:, 0] >= 0) & (uv[:, 0] < w) &
            (uv[:, 1] >= 0) & (uv[:, 1] < h)
        )
        return int(valid_uv.sum()), uv, pts_cam, valid_uv

    score_fwd, uv_fwd, pts_cam_fwd, valid_fwd = score(T_candidate)
    score_inv, uv_inv, pts_cam_inv, valid_inv = score(np.linalg.inv(T_candidate))

    if score_inv > score_fwd:
        print(f'Using inverse extrinsic direction: valid_in_image={score_inv} vs forward={score_fwd}')
        return np.linalg.inv(T_candidate), uv_inv, pts_cam_inv, valid_inv, 'inverse'

    print(f'Using forward extrinsic direction: valid_in_image={score_fwd} vs inverse={score_inv}')
    return T_candidate, uv_fwd, pts_cam_fwd, valid_fwd, 'forward'


def recover_t0_pose_global_enu(clip_dir: Path, t0_sod: float, num_history_steps: int, time_step: float):
    ego_log_path = clip_dir / 'ego_pos.log'
    pose_times_sod, pose_xyz_global_enu, pose_rot_global_enu = _load_ego_pose_log(ego_log_path)

    history_offsets = np.arange(
        -(num_history_steps - 1) * time_step,
        time_step / 2,
        time_step,
        dtype=np.float64,
    )
    history_times_sod = t0_sod + history_offsets

    ego_history_xyz_global_enu, ego_history_rot_global_enu = _lookup_pose_samples(
        pose_times_sod,
        pose_xyz_global_enu,
        pose_rot_global_enu,
        history_times_sod,
    )

    t0_xyz_global_enu = ego_history_xyz_global_enu[-1].copy()
    t0_rot_global_enu = spt.Rotation.from_matrix(ego_history_rot_global_enu[-1])
    return t0_xyz_global_enu, t0_rot_global_enu


def plotting_frame_to_t0_local(xyz_plot: np.ndarray, eval_xy_rotation_deg: float) -> np.ndarray:
    return rotate_xyz_xy_plane(xyz_plot, -eval_xy_rotation_deg)


def t0_local_to_global_enu(xyz_t0_local: np.ndarray, t0_xyz_global_enu: np.ndarray, t0_rot_global_enu: spt.Rotation) -> np.ndarray:
    return t0_rot_global_enu.apply(xyz_t0_local) + t0_xyz_global_enu


### Load calibration from repo_root/calibration/


In [ ]:
for p in [FRONT_K_PATH, FRONT_K_ORIG_PATH, FRONT_EXT_PATH, FRONT_DIST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing calibration file: {p}')

K_front = load_matrix_txt(FRONT_K_PATH)
K_front_orig = load_matrix_txt(FRONT_K_ORIG_PATH)
T_front = load_matrix_txt(FRONT_EXT_PATH)
dist_front = load_matrix_txt(FRONT_DIST_PATH)

print('K_front =\n', K_front)
print('K_front_orig =\n', K_front_orig)
print('T_front =\n', T_front)
print('dist_front =\n', dist_front)

print('\nNOTE: Distortion is loaded but not applied in this notebook.')


### Build clip cache once


In [ ]:
clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=True, force_rebuild=False)
print('clip cache ready')
print('num cameras =', len(clip_cache.camera_paths))
print('num pose samples =', len(clip_cache.pose_times_sod))


### Load model and processor


In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Model and processor loaded.')


### Load data and run offline inference


In [ ]:
data = load_offline_dataset(
    clip_dir=CLIP_DIR,
    t0_sod=T0_SOD,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    debug=True,
    image_size=IMAGE_SIZE,
)

result = run_offline_inference_window(
    data=data,
    model=model,
    processor=processor,
    device=DEVICE,
    top_p=TOP_P,
    temperature=TEMPERATURE,
    num_traj_samples=NUM_TRAJ_SAMPLES,
    max_generation_length=MAX_GENERATION_LENGTH,
    time_step=TIME_STEP,
    eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
)

print('Inference done.')
print('Chain-of-Causation:')
print(result['cot_value'])
display(result['df_metrics'])


### Load original front camera frame


In [ ]:
front_cam_row = pick_front_camera_row(data, CLIP_DIR, row_override=FRONT_CAMERA_ROW_OVERRIDE)
print('front_cam_row =', front_cam_row)

camera_paths = helper.discover_offline_camera_files(CLIP_DIR)
camera_paths_sorted = sorted(camera_paths, key=lambda p: helper.infer_camera_index(p.name))
front_video_path = camera_paths_sorted[front_cam_row]
print('front_video_path =', front_video_path)

requested_front_frame_idx = int(data['actual_video_frame_indices'][front_cam_row, FRONT_FRAME_T].item())
requested_front_ts = float(data['absolute_timestamps_sod'][front_cam_row, FRONT_FRAME_T].item())

front_img_orig = decode_single_frame(front_video_path, requested_front_frame_idx)
image_h, image_w = front_img_orig.shape[:2]

print('requested_front_frame_idx =', requested_front_frame_idx)
print('requested_front_ts =', requested_front_ts)
print('front_img_orig shape =', front_img_orig.shape)


### Prepare History / GT / Pred in global ENU


In [ ]:
t0_xyz_global_enu, t0_rot_global_enu = recover_t0_pose_global_enu(
    clip_dir=CLIP_DIR,
    t0_sod=T0_SOD,
    num_history_steps=NUM_HISTORY_STEPS,
    time_step=TIME_STEP,
)

hist_xyz_t0_local = plotting_frame_to_t0_local(result['hist_xyz_plot'], EVAL_XY_ROTATION_DEG)
gt_xyz_t0_local = plotting_frame_to_t0_local(result['gt_xyz_plot'], EVAL_XY_ROTATION_DEG)
pred_xyz_t0_local = plotting_frame_to_t0_local(result['pred_best_xyz'], EVAL_XY_ROTATION_DEG)

hist_xyz_global_enu = t0_local_to_global_enu(hist_xyz_t0_local, t0_xyz_global_enu, t0_rot_global_enu)
gt_xyz_global_enu = t0_local_to_global_enu(gt_xyz_t0_local, t0_xyz_global_enu, t0_rot_global_enu)
pred_xyz_global_enu = t0_local_to_global_enu(pred_xyz_t0_local, t0_xyz_global_enu, t0_rot_global_enu)

print('Prepared history/gt/pred in global ENU.')


### Choose extrinsic direction heuristically


In [ ]:
T_front_used, uv_pred_probe, pred_cam_probe, valid_pred_probe, extrinsic_mode = choose_reasonable_extrinsic_direction(
    pred_xyz_global_enu,
    T_front,
    K_front_orig,
    (image_h, image_w),
)

print('extrinsic_mode =', extrinsic_mode)


### Project History / GT / Pred onto front camera image


In [ ]:
hist_xyz_cam = transform_points_homogeneous(hist_xyz_global_enu, T_front_used)
gt_xyz_cam = transform_points_homogeneous(gt_xyz_global_enu, T_front_used)
pred_xyz_cam = transform_points_homogeneous(pred_xyz_global_enu, T_front_used)

uv_hist, valid_hist = project_points_pinhole(hist_xyz_cam, K_front_orig)
uv_gt, valid_gt = project_points_pinhole(gt_xyz_cam, K_front_orig)
uv_pred, valid_pred = project_points_pinhole(pred_xyz_cam, K_front_orig)

print('valid_hist =', int(valid_hist.sum()), '/', len(valid_hist))
print('valid_gt   =', int(valid_gt.sum()), '/', len(valid_gt))
print('valid_pred =', int(valid_pred.sum()), '/', len(valid_pred))


### Side-by-side: original image projection + BEV trajectory plot


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left: original image projection
ax = axes[0]
ax.imshow(front_img_orig)

if valid_hist.any():
    ax.plot(
        uv_hist[valid_hist, 0],
        uv_hist[valid_hist, 1],
        'b-o',
        linewidth=2.0,
        markersize=4,
        label='History',
    )

if valid_gt.any():
    ax.plot(
        uv_gt[valid_gt, 0],
        uv_gt[valid_gt, 1],
        'k-o',
        linewidth=2.5,
        markersize=4,
        label='GT',
    )

if valid_pred.any():
    ax.plot(
        uv_pred[valid_pred, 0],
        uv_pred[valid_pred, 1],
        'r-o',
        linewidth=2.5,
        markersize=4,
        label='Pred',
    )

ax.set_xlim(0, image_w)
ax.set_ylim(image_h, 0)
ax.set_title(f'Front Camera Projection | extrinsic_mode={extrinsic_mode}')
ax.legend()

# Right: BEV plot in current validated plotting frame
ax = axes[1]

hist_bev = result['hist_xyz_plot']
gt_bev = result['gt_xyz_plot']
pred_bev = result['pred_best_xyz']

xlim, ylim = _compute_adaptive_xy_limits(hist_bev, gt_bev, pred_bev, min_span=20.0, pad_ratio=0.12, pad_min=2.0)

ax.plot(hist_bev[:, 0], hist_bev[:, 1], 'b-o', linewidth=2, markersize=3, label='History')
ax.plot(gt_bev[:, 0], gt_bev[:, 1], 'k-o', linewidth=2.5, markersize=3.5, label='GT')
ax.plot(pred_bev[:, 0], pred_bev[:, 1], 'r-o', linewidth=2.5, markersize=3.5, label='Pred')
ax.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('BEV trajectory plot')
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='box')
ax.legend()

plt.tight_layout()
plt.show()


### Debug table of projected points


In [ ]:
df_proj_debug = pd.DataFrame({
    'hist_u': uv_hist[:, 0],
    'hist_v': uv_hist[:, 1],
    'hist_valid': valid_hist,
    'gt_u': uv_gt[:, 0],
    'gt_v': uv_gt[:, 1],
    'gt_valid': valid_gt,
    'pred_u': uv_pred[:, 0],
    'pred_v': uv_pred[:, 1],
    'pred_valid': valid_pred,
})

display(df_proj_debug)
